# Our approach — DeepSeek-V3 on StrategyClaim (500 claims, annotators' facts)

The 500-claim experimental split of the ArgLLM repository, with the annotators' facts. Pipeline and results. Results: credulous 89.0, skeptical 89.2, net support 88.6, 16.4% of claims with more than one extension (17 + 38 + 27 of 500).

Run on Kaggle; the API key is read from Kaggle secrets (`DEEPSEEK_API_KEY`). Outputs are kept so the reported numbers can be checked.

In [1]:
!pip install -q openai datasets scikit-learn

import os, re, json, time, pickle
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

WORK_DIR = Path("/kaggle/working/deepseek_strategyclaim_v2")
WORK_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"),
                base_url="https://api.deepseek.com")
MODEL_NAME = "deepseek-chat"

# Same StrategyClaim data as Version 1 (ArgLLM repository, 500 balanced)
if not Path("/kaggle/working/argumentative-llms").exists():
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk("/kaggle/working/argumentative-llms/Datasets/StrategyQA/Experiment").to_pandas().reset_index(drop=True)

def format_facts(facts):
    if facts is None: return ""
    if isinstance(facts, (list, tuple, np.ndarray)):
        return "\n".join(f"- {str(f)}" for f in facts)
    return str(facts)

baf_df = pd.DataFrame({
    "example_index": range(len(df)),
    "claim": df["claim"].astype(str),
    "facts_text": df["facts"].apply(format_facts),
    "gold": df["valid"].apply(lambda v: int(bool(v))),
})
print("Loaded:", len(baf_df), "| True:", baf_df["gold"].sum(), "| first:", baf_df.iloc[0]["claim"][:60])

Loaded: 500 | True: 250 | first: Number of Imams Reza Shah believed in exceed number of Jesus


In [2]:
BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5, "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55, "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55, "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2, "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0, "max_retries": 5,
}

def clip01(v):
    try: return max(0.0, min(1.0, float(v)))
    except: return 0.0

def extract_json_object(text):
    text = str(text or "").strip(); text = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    s = text.find("{"); e = text.rfind("}")
    if s >= 0 and e > s: return json.loads(text[s:e+1])
    raise ValueError("No JSON")

def json_call(prompt, temperature, max_tokens, retries=5):
    last = None
    for a in range(1, retries+1):
        try:
            resp = client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You are an expert in claim verification, formal argumentation and strict JSON output."},
                          {"role":"user","content":prompt}],
                temperature=temperature, max_tokens=max_tokens)
            t = resp.choices[0].message.content
            if not t: raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last = e
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(12); continue
            if a < retries: time.sleep(3*a)
    raise RuntimeError(f"failed: {last}")

def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw = json_call(prompt, config["generation_temperature"], 1900, config["max_retries"])
    out = {"interpreted_claim": str(raw.get("interpreted_claim","")).strip(), "pro": [], "con": []}
    for role in ("pro","con"):
        for e in (raw.get(role,[]) if isinstance(raw.get(role,[]),list) else []):
            if not isinstance(e,dict): continue
            t = str(e.get("text","")).strip()
            if not t: continue
            out[role].append({"text":t,"relevance":clip01(e.get("relevance")),"grounding":clip01(e.get("grounding")),
                              "clarity":clip01(e.get("clarity")),"overall_quality":clip01(e.get("overall_quality"))})
    return out

def select_arguments(gen, config=BAF_CONFIG):
    nodes = {"CLAIM":{"role":"CLAIM","text":""}}; scored = []
    for src,role,pre,mx in [("pro","PRO","P",config["max_pro_keep"]),("con","CON","N",config["max_con_keep"])]:
        cands = []
        for it in gen.get(src,[]):
            rl,g = clip01(it.get("relevance")),clip01(it.get("grounding")); cl,q = clip01(it.get("clarity")),clip01(it.get("overall_quality"))
            rank = 0.30*rl+0.30*g+0.15*cl+0.25*q; sc = {**it,"role":role,"rank_score":rank}; scored.append(sc)
            if rl>=config["min_relevance"] and g>=config["min_grounding"] and cl>=config["min_clarity"] and q>=config["min_quality"]: cands.append(sc)
        cands.sort(key=lambda x:x["rank_score"], reverse=True)
        for n,it in enumerate(cands[:mx],1):
            nodes[f"{pre}{n}"] = {"role":role,"text":it["text"],"relevance":it["relevance"],"grounding":it["grounding"],
                                  "clarity":it["clarity"],"overall_quality":it["overall_quality"],"rank_score":it["rank_score"]}
    return {"nodes":nodes,"all_scored":scored}

def build_relation_candidates(nodes):
    pro = [n for n,i in nodes.items() if i["role"]=="PRO"]; con = [n for n,i in nodes.items() if i["role"]=="CON"]; c = []
    for p in pro: c.append({"source":p,"target":"CLAIM","allowed":["SUPPORT","NONE"]})
    for n in con: c.append({"source":n,"target":"CLAIM","allowed":["ATTACK","NONE"]})
    for p in pro:
        for n in con:
            c.append({"source":p,"target":n,"allowed":["ATTACK","NONE"]}); c.append({"source":n,"target":p,"allowed":["ATTACK","NONE"]})
    return c
print("✅ CELL 2 ready")

✅ CELL 2 ready


In [3]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG, batch_size=16):
    cands = build_relation_candidates(nodes)
    at = "\n".join(f"{nid} [{i['role']}]: {i['text']}" for nid,i in nodes.items() if nid!="CLAIM")
    aud = []
    for bs in range(0, len(cands), batch_size):
        b = cands[bs:bs+batch_size]
        pb = "\n".join(f"{it['source']} -> {it['target']} | allowed={it['allowed']}" for it in b)
        prompt = f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw = json_call(prompt, config["audit_temperature"], 2200, config["max_retries"])
        exp = {(it["source"],it["target"]) for it in b}; allow = {(it["source"],it["target"]):set(it["allowed"]) for it in b}; ret = {}
        for r in (raw.get("relations",[]) if isinstance(raw.get("relations",[]),list) else []):
            if not isinstance(r,dict): continue
            s,t = str(r.get("source","")).strip(), str(r.get("target","")).strip()
            typ = str(r.get("type","NONE")).strip().upper(); pr = (s,t)
            if pr not in exp or typ not in allow[pr]: continue
            ret[pr] = {"source":s,"target":t,"type":typ,"target_match":clip01(r.get("target_match")),"grounding":clip01(r.get("grounding")),
                       "relation_score":clip01(r.get("relation_score")),"reason":str(r.get("reason","")).strip()}
        for it in b:
            pr = (it["source"],it["target"])
            if pr not in ret:
                ret[pr] = {"source":pr[0],"target":pr[1],"type":"NONE","target_match":0.0,"grounding":0.0,"relation_score":0.0,"reason":"omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations":aud}

def relation_passes(r, config=BAF_CONFIG):
    return r["target_match"]>=config["min_target_match"] and r["grounding"]>=config["min_grounding_edge"] and r["relation_score"]>=config["min_relation_score"]
def relation_lookup(rels): return {(r["source"],r["target"]):r for r in rels}
def select_formal_supports(rels, config=BAF_CONFIG):
    return sorted(set((r["source"],r["target"]) for r in rels if r["type"]=="SUPPORT" and r["target"]=="CLAIM" and r["source"].startswith("P") and relation_passes(r,config)))
def select_con_claim_attacks(rels, config=BAF_CONFIG):
    p = [r for r in rels if r["type"]=="ATTACK" and r["target"]=="CLAIM" and r["source"].startswith("N") and relation_passes(r,config)]
    p.sort(key=lambda x:x["relation_score"], reverse=True); return p[:config["max_claim_attacks"]]
def select_independent_conflict_pairs(rels, config=BAF_CONFIG):
    lk = relation_lookup(rels); cand = []
    pro = sorted({r["source"] for r in rels if r["source"].startswith("P")}); con = sorted({r["source"] for r in rels if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc,cp = lk.get((p,n)), lk.get((n,p))
            if not pc or not cp: continue
            if pc["type"]=="ATTACK" and cp["type"]=="ATTACK" and relation_passes(pc,config) and relation_passes(cp,config):
                cand.append({"pro":p,"con":n,"pair_score":min(pc["relation_score"],cp["relation_score"])})
    cand.sort(key=lambda x:x["pair_score"], reverse=True); sel = []; up = set(); uc = set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc: continue
        sel.append(c); up.add(c["pro"]); uc.add(c["con"])
        if len(sel) >= config["max_conflict_pairs"]: break
    return sel, cand
def support_closure(nodes, sup):
    cl = set(sup); ch = True
    while ch:
        ch = False; new = set()
        for a,b in cl:
            for c,d in cl:
                if b==c and (a,d) not in cl: new.add((a,d))
        if new: cl |= new; ch = True
    return cl
def derive_attacks(nodes, direct, sup):
    sc = support_closure(nodes, sup); sa = set(); se = set()
    for s,sd in sc:
        for at,t in direct:
            if sd==at: sa.add((s,t))
    for at,ad in direct:
        for s,sd in sc:
            if ad==s: se.add((at,sd))
    eff = {e for e in (set(direct)|sa|se) if e[0]!=e[1]}
    return {"support_closure":sorted(sc),"supported_attacks":sorted(sa),"secondary_attacks":sorted(se),"effective_attacks":sorted(eff)}
print("✅ CELL 3 ready")

✅ CELL 3 ready


In [4]:
def all_subsets(items):
    il = sorted(items)
    for sz in range(len(il)+1):
        for sub in combinations(il, sz): yield set(sub)
def cf(e, att): return not any(s in e and t in e for s,t in att)
def att_of(a, att): return {s for s,t in att if t==a}
def ext_att(e, a, att): return any(s in e and t==a for s,t in att)
def defends(e, a, att): return all(ext_att(e,x,att) for x in att_of(a,att))
def admissible(e, att): return cf(e,att) and all(defends(e,a,att) for a in e)
def preferred_ext(args, att):
    adm = [s for s in all_subsets(args) if admissible(s,att)]
    return [c for c in adm if not any(c<o for o in adm)]
def stable_ext(args, att):
    return [c for c in all_subsets(args) if cf(c,att) and all(ext_att(c,a,att) for a in args-c)]
def norm(exts): return [sorted(e) for e in exts]
def decisions(exts, cid="CLAIM"):
    if not exts: return {"credulous":0,"skeptical":0,"majority":0}
    cc = sum(cid in e for e in exts)
    return {"credulous":int(cc>=1),"skeptical":int(cc==len(exts)),"majority":int(cc*2>=len(exts))}

def rebuild(inter, config=BAF_CONFIG):
    nodes = dict(inter["nodes"]); rels = inter["all_audited_relations"]; sup = set(select_formal_supports(rels,config))
    sca = select_con_claim_attacks(rels,config); sp,_ = select_independent_conflict_pairs(rels,config); direct = set()
    for r in sca: direct.add((r["source"],r["target"]))
    for p in sp: direct.add((p["pro"],p["con"])); direct.add((p["con"],p["pro"]))
    conn = {"CLAIM"}
    for st in sup|direct: conn.add(st[0]); conn.add(st[1])
    nodes = {k:v for k,v in nodes.items() if k in conn}; vids = set(nodes)
    sup = {e for e in sup if e[0] in vids and e[1] in vids}; direct = {e for e in direct if e[0] in vids and e[1] in vids}
    der = derive_attacks(vids, direct, sup); eff = set(map(tuple, der["effective_attacks"]))
    pref = preferred_ext(vids, eff); stab = stable_ext(vids, eff)
    return {**inter, "nodes":nodes, "supports":sorted(sup), "selected_claim_attacks":sca,
        "selected_conflict_pairs":[{"pro":p["pro"],"con":p["con"],"pair_score":p["pair_score"]} for p in sp],
        "direct_attacks":sorted(direct), "support_closure":der["support_closure"], "supported_attacks":der["supported_attacks"],
        "secondary_attacks":der["secondary_attacks"], "effective_attacks":der["effective_attacks"],
        "preferred_extensions":norm(pref), "stable_extensions":norm(stab),
        "preferred_decisions":decisions(pref), "stable_decisions":decisions(stab)}

def run_example(row, config=BAF_CONFIG):
    claim = str(row["claim"]).strip(); facts = str(row["facts_text"]).strip(); gold = int(row["gold"])
    gen = generate_and_score_arguments(claim, facts, config); sel = select_arguments(gen, config)
    nodes = sel["nodes"]; nodes["CLAIM"]["text"] = claim
    audit = audit_baf_relations(claim, facts, nodes, config, config["audit_batch_size"])
    inter = {"example_index":int(row["example_index"]), "claim":claim, "facts_text":facts, "gold":gold,
             "interpreted_claim":gen["interpreted_claim"], "generated_arguments":gen, "argument_scores":sel["all_scored"],
             "nodes":nodes, "all_audited_relations":audit["all_audited_relations"], "model":MODEL_NAME,
             "pipeline_version":"v2", "config":dict(config)}
    return rebuild(inter, config)

test = run_example(baf_df.iloc[0])
print("gold", test["gold"], "| credulous", test["preferred_decisions"]["credulous"],
      "| nodes", len(test["nodes"]), "| extensions", len(test["preferred_extensions"]))

gold 0 | credulous 1 | nodes 7 | extensions 8


In [5]:
CKPT = WORK_DIR / "deepseek_strategyclaim_v2_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))

def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)}...")
    for k, idx in enumerate(todo, 1):
        try:
            res = run_example(baf_df.iloc[idx])
        except Exception as e:
            print(f"error at {idx}: {str(e)[:60]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 20 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} -> {len(cur)} total | running credulous acc {acc*100:.1f}%")
        time.sleep(0.3)
    print(f"done: {len(load_ck())}/{len(baf_df)}")

run_batch(500)

Have 0/500. Running 500...
  20/500 -> 20 total | running credulous acc 90.0%
  40/500 -> 40 total | running credulous acc 92.5%
  60/500 -> 60 total | running credulous acc 90.0%
  80/500 -> 80 total | running credulous acc 90.0%
  100/500 -> 100 total | running credulous acc 89.0%
  120/500 -> 120 total | running credulous acc 89.2%
  140/500 -> 140 total | running credulous acc 88.6%
  160/500 -> 160 total | running credulous acc 89.4%
  180/500 -> 180 total | running credulous acc 88.9%
  200/500 -> 200 total | running credulous acc 88.5%
  220/500 -> 220 total | running credulous acc 89.5%
  240/500 -> 240 total | running credulous acc 89.6%
  260/500 -> 260 total | running credulous acc 90.4%
  280/500 -> 280 total | running credulous acc 90.4%
  300/500 -> 300 total | running credulous acc 89.3%
  320/500 -> 320 total | running credulous acc 89.4%
  340/500 -> 340 total | running credulous acc 89.7%
  360/500 -> 360 total | running credulous acc 89.7%
  380/500 -> 380 total | ru

In [6]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]; n = len(results)

def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    return {"method":name, "n":n, "accuracy":round(accuracy_score(gold,pred)*100,1),
            "precision":round(precision_score(gold,pred,zero_division=0)*100,1),
            "recall":round(recall_score(gold,pred,zero_division=0)*100,1),
            "f1":round(f1_score(gold,pred,zero_division=0)*100,1), "TN":tn,"FP":fp,"FN":fn,"TP":tp}
def no_objection_pred(r, exts_key):
    exts = [set(e) for e in r[exts_key]]
    if not exts or not any("CLAIM" in e for e in exts): return 0
    attackers = {s for (s,t) in map(tuple, r["direct_attacks"]) if t=="CLAIM"}
    return int(not (attackers & set().union(*exts)))
def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))

rows = []
for sem in ["preferred","stable"]:
    for rule in ["credulous","skeptical","majority"]:
        rows.append(metrics_row(f"{sem}_{rule}", [r[f"{sem}_decisions"][rule] for r in results]))
    rows.append(metrics_row(f"{sem}_no_objection", [no_objection_pred(r, f"{sem}_extensions") for r in results]))
rows.append(metrics_row("net_support", [net_support_pred(r) for r in results]))
pd.set_option("display.width", 220)
print(f"=== DeepSeek StrategyClaim — Version 2 ({n} examples) ===\n")
print(pd.DataFrame(rows).to_string())

from collections import Counter
print("\nextension counts:", dict(sorted(Counter(len(r["preferred_extensions"]) for r in results).items())))
print("failed examples:", 500 - n)

pickle.dump(results, open(WORK_DIR/f"deepseek_strategyclaim_v2_{n}.pkl","wb"))
print("saved:", WORK_DIR/f"deepseek_strategyclaim_v2_{n}.pkl")

=== DeepSeek StrategyClaim — Version 2 (500 examples) ===

                   method    n  accuracy  precision  recall    f1   TN  FP  FN   TP
0     preferred_credulous  500      89.0       83.3    97.6  89.9  201  49   6  244
1     preferred_skeptical  500      89.2       95.0    82.8  88.5  239  11  43  207
2      preferred_majority  500      89.2       93.0    84.8  88.7  234  16  38  212
3  preferred_no_objection  500      89.2       95.0    82.8  88.5  239  11  43  207
4        stable_credulous  500      89.0       83.3    97.6  89.9  201  49   6  244
5        stable_skeptical  500      89.2       95.0    82.8  88.5  239  11  43  207
6         stable_majority  500      89.2       93.0    84.8  88.7  234  16  38  212
7     stable_no_objection  500      89.2       95.0    82.8  88.5  239  11  43  207
8             net_support  500      88.6       82.5    98.0  89.6  198  52   5  245

extension counts: {1: 418, 2: 17, 4: 38, 8: 27}
failed examples: 0
saved: /kaggle/working/deepseek_s